# PerturbQA 数据探索

加载 PerturbQA 基准数据、知识图谱、基因摘要和模型输出，进行探索性分析。

In [ ]:
import sys
sys.path.insert(0, '..')
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# 数据目录
benchmark_dir = Path('../data/benchmark')
kg_dir = Path('../data/knowledge_graph')
gene_dir = Path('../data/gene_summaries')
model_dir = Path('../data/model_outputs')
results_dir = Path('../data/results')

print('数据目录检查:')
for d in [benchmark_dir, kg_dir, gene_dir, model_dir, results_dir]:
    exists = d.exists()
    n_files = len(list(d.rglob('*'))) if exists else 0
    print(f'  {d}: exists={exists}, files={n_files}')

In [ ]:
# 加载 DE 任务数据 (k562)
de_dir = benchmark_dir / 'de'
if de_dir.exists():
    de_files = list(de_dir.glob('*k562*'))
    print(f'DE k562 files: {[f.name for f in de_files]}')
    
    if de_files:
        f = de_files[0]
        if f.suffix == '.jsonl':
            data = [json.loads(line) for line in open(f) if line.strip()]
        elif f.suffix == '.json':
            data = json.load(open(f))
            if isinstance(data, dict):
                data = next((v for v in data.values() if isinstance(v, list)), [])
        
        print(f'\nLoaded: {f.name}')
        print(f'Samples: {len(data)}')
        if data:
            print(f'Fields: {list(data[0].keys())}')
            print(f'\nFirst sample:')
            for k, v in data[0].items():
                print(f'  {k}: {str(v)[:100]}')

In [ ]:
# 统计标签分布和扰动基因数量
if 'data' in locals() and data:
    # 找标签字段
    label_key = None
    pert_key = None
    for key in data[0].keys():
        if 'label' in key.lower() or 'target' in key.lower():
            label_key = key
        if 'pert' in key.lower() or ('gene' in key.lower() and 'label' not in key.lower()):
            pert_key = key
    
    if label_key:
        labels = [d.get(label_key) for d in data]
        print(f'Label distribution ({label_key}):')
        for label, count in Counter(labels).most_common():
            print(f'  {label}: {count}')
    
    if pert_key:
        perts = [str(d.get(pert_key)) for d in data]
        print(f'\nUnique perturbation genes: {len(set(perts))}')
        print(f'Top 10:')
        for p, c in Counter(perts).most_common(10):
            print(f'  {p}: {c}')

In [ ]:
# 检查知识图谱
if kg_dir.exists():
    kg_files = list(kg_dir.rglob('*'))
    kg_files = [f for f in kg_files if f.is_file() and not f.name.startswith('.')]
    print(f'Knowledge graph files: {len(kg_files)}')
    for f in sorted(kg_files)[:10]:
        print(f'  {f.relative_to(kg_dir)}: {f.stat().st_size / 1024:.1f} KB')

In [ ]:
# 检查模型输出
if model_dir.exists():
    model_files = list(model_dir.rglob('*'))
    model_files = [f for f in model_files if f.is_file() and f.suffix in ['.json', '.jsonl', '.csv']]
    print(f'Model output files: {len(model_files)}')
    
    # 分类
    summer = [f for f in model_files if 'summer' in f.name.lower()]
    nocot = [f for f in model_files if 'nocot' in f.name.lower()]
    noret = [f for f in model_files if 'noretrieve' in f.name.lower()]
    
    print(f'  Summer outputs: {len(summer)}')
    print(f'  No-CoT outputs: {len(nocot)}')
    print(f'  No-Retrieval outputs: {len(noret)}')

In [ ]:
# 计算随机预测的 macro AUROC (示例)
from sklearn.metrics import roc_auc_score
import numpy as np

if 'data' in locals() and data and label_key:
    labels = np.array([1 if str(d.get(label_key)).lower() in ['1', 'true', 'positive', 'up'] else 0 for d in data])
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    
    # 随机预测
    np.random.seed(42)
    random_scores = np.random.rand(len(labels))
    
    try:
        auroc = roc_auc_score(labels, random_scores)
        print(f'Random prediction macro AUROC: {auroc:.4f}')
        print(f'Positive samples: {n_pos}')
        print(f'Negative samples: {n_neg}')
    except Exception as e:
        print(f'AUROC calculation failed: {e}')
        print(f'Labels: {Counter(labels)}')

## 数据类型区分

| 类型 | 说明 | 位置 |
|------|------|------|
| 原始单细胞 CRISPRi 表达数据 | 各论文原始数据 | 不包含在本项目 |
| PerturbQA 加工后的问答标签 | DE/direction/GSE 基准 | data/benchmark/ |
| 生物知识图谱 | 基因-基因/通路关系 | data/knowledge_graph/ |
| 基因自然语言摘要 | 基因功能描述 | data/gene_summaries/ |
| Summer 及消融模型输出 | 模型预测结果 | data/model_outputs/ |
| 最终评价结果 | 评估指标 | data/results/ |